# Human / Author Review — Golden Dataset

این Notebook برای مرور انسانی و تأیید نهایی مؤلف ساخته شده است.

منبع حقیقت این Notebook فایل `human_review_registry.json` است و داده‌های Review در خود این Notebook نگهداری نمی‌شوند.

برای مشاهده آخرین وضعیت، کافی است Notebook را Run کنید.

In [ ]:

from pathlib import Path
import json


def find_registry():
    candidates = []

    cwd = Path.cwd().resolve()

    candidates.extend([
        cwd
        / "data"
        / "phase2"
        / "human_review"
        / "human_review_registry.json",

        cwd
        / ".."
        / ".."
        / "data"
        / "phase2"
        / "human_review"
        / "human_review_registry.json",
    ])

    for parent in [cwd] + list(cwd.parents):
        candidates.append(
            parent
            / "data"
            / "phase2"
            / "human_review"
            / "human_review_registry.json"
        )

    checked = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if str(candidate) in checked:
            continue

        checked.add(str(candidate))

        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "human_review_registry.json was not found."
    )


REGISTRY_PATH = find_registry()

review_registry = json.loads(
    REGISTRY_PATH.read_text(
        encoding="utf-8"
    )
)

print("Registry:", REGISTRY_PATH)

print(
    "Completed review cases:",
    len(
        review_registry.get(
            "completed_human_review_cases",
            []
        )
    )
)

print(
    "Active Pilot:",
    (
        review_registry.get(
            "active_workspace"
        )
        or {}
    ).get(
        "pilot_case_id"
    )
)

print(
    "Remaining Pilots:",
    review_registry.get(
        "remaining_pilot_count"
    )
)


In [ ]:

from IPython.display import display, Markdown


def show_text(value):
    if value is None:
        return "—"

    return str(value)


def show_evidence_parts(parts):
    for part in parts or []:

        if not isinstance(part, dict):
            continue

        text = part.get("text")

        if not text:
            continue

        display(
            Markdown(
                "> "
                +
                show_text(text).replace(
                    "\n",
                    "\n> "
                )
            )
        )


completed_cases = review_registry.get(
    "completed_human_review_cases",
    []
)


display(
    Markdown(
        "# موارد تکمیل‌شده برای مرور مؤلف"
    )
)


for index, case in enumerate(
    completed_cases,
    start=1,
):

    display(
        Markdown(
            f"## Case {index}"
        )
    )


    display(
        Markdown(
            "### متن اصلی منبع"
        )
    )


    evidence_parts = (
        case.get(
            "source_evidence"
        )
        or
        case.get(
            "source_evidence_snapshot"
        )
        or
        []
    )


    show_evidence_parts(
        evidence_parts
    )


    drafts = case.get(
        "model_drafts",
        []
    )


    if drafts:

        display(
            Markdown(
                "### تاریخچه سؤال"
            )
        )


        for draft in drafts:

            display(
                Markdown(
                    "**Revision "
                    +
                    show_text(
                        draft.get(
                            "revision"
                        )
                    )
                    +
                    "**  \n"
                    +
                    show_text(
                        draft.get(
                            "question_text"
                        )
                    )
                    +
                    "  \nHuman Decision: **"
                    +
                    show_text(
                        draft.get(
                            "human_decision"
                        )
                    )
                    +
                    "**"
                )
            )


    answer_review = case.get(
        "expected_answer_review"
    )


    if isinstance(
        answer_review,
        dict,
    ):

        display(
            Markdown(
                "### پاسخ پیشنهادی"
            )
        )

        display(
            Markdown(
                show_text(
                    answer_review.get(
                        "expected_answer_text"
                    )
                )
                +
                "  \nHuman Decision: **"
                +
                show_text(
                    answer_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    citation_review = case.get(
        "citation_review"
    )


    if isinstance(
        citation_review,
        dict,
    ):

        display(
            Markdown(
                "### Citation"
            )
        )

        display(
            Markdown(
                "Human Decision: **"
                +
                show_text(
                    citation_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    author_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )


    display(
        Markdown(
            "### تأیید نهایی مؤلف\n"
            "Status: **"
            +
            show_text(
                author_review.get(
                    "status"
                )
            )
            +
            "**"
        )
    )


active_case = review_registry.get(
    "active_workspace"
)


if isinstance(
    active_case,
    dict,
):

    display(
        Markdown(
            "# Case فعال"
        )
    )


    display(
        Markdown(
            "### متن اصلی منبع"
        )
    )


    show_evidence_parts(
        active_case.get(
            "source_evidence"
        )
        or
        []
    )


    drafts = active_case.get(
        "model_drafts",
        []
    )


    for draft in drafts:

        display(
            Markdown(
                "### سؤال\n"
                +
                show_text(
                    draft.get(
                        "question_text"
                    )
                )
                +
                "\n\nHuman Decision: **"
                +
                show_text(
                    draft.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    answer_review = active_case.get(
        "expected_answer_review"
    )


    if isinstance(
        answer_review,
        dict,
    ):

        display(
            Markdown(
                "### پاسخ پیشنهادی\n"
                +
                show_text(
                    answer_review.get(
                        "expected_answer_text"
                    )
                )
                +
                "\n\nHuman Decision: **"
                +
                show_text(
                    answer_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    display(
        Markdown(
            "### تأیید نهایی مؤلف\n"
            "Status: **"
            +
            show_text(
                (
                    active_case.get(
                        "author_final_review"
                    )
                    or
                    {}
                ).get(
                    "status"
                )
            )
            +
            "**"
        )
    )
